# LSTM Experiments & Evaluation

Notebook ini:
1. Load bobot LSTM terbaik dari tiap variasi training
2. Evaluasi BLEU-4 dan METEOR pada test set
3. Bandingkan Keras vs From Scratch inference
4. Qualitative analysis: 10 contoh gambar
5. Pengaruh max caption length

In [ ]:
import os, sys, json, time, pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../..')))

import tensorflow as tf
from tensorflow import keras

from src.lstm.train_keras import build_lstm_decoder, EMBED_DIM
from src.lstm.model import LSTMDecoder
from src.shared.caption_utils import load_vocabulary, clean_caption
from src.shared.metrics import bleu4, meteor

print('TF:', tf.__version__)

## 1. Load Data & Vocabulary

In [ ]:
FEATURES_PATH = '../../features/flickr8k_features.npy'
FEAT_IDX_PATH = '../../features/flickr8k_idx.json'
VOCAB_PATH    = '../../features/vocab.json'
CAPTIONS_TXT  = '../../data/flickr8k/captions.txt'
IMAGES_DIR    = '../../data/flickr8k/Images/'
MODELS_DIR    = '../../models/lstm/'

features = np.load(FEATURES_PATH)
with open(FEAT_IDX_PATH) as f:
    img_idx = json.load(f)

image_ids  = list(img_idx.keys())
vocab      = load_vocabulary(VOCAB_PATH)
VOCAB_SIZE = len(vocab)
FEATURE_DIM = features.shape[1]
idx2word   = {v: k for k, v in vocab.items()}

print(f'Features: {features.shape}, Vocab: {VOCAB_SIZE}')

captions_dict = {}
with open(CAPTIONS_TXT) as f:
    for line in f:
        line = line.strip()
        if not line or line.lower().startswith('image'):
            continue
        parts = line.split(',', 1)
        if len(parts) < 2:
            continue
        img_file, caption = parts
        img_id = img_file.split('#')[0].strip()
        clean  = clean_caption(caption)
        captions_dict.setdefault(img_id, []).append(clean)

test_ids = image_ids[7000:8000]
test_captions = {k: captions_dict.get(k, []) for k in test_ids}
print(f'Test images: {len(test_ids)}')

## 2. Fungsi Helper Evaluasi

In [ ]:
MAX_LEN = 30

CONFIGS = [
    {'name': 'lstm-1layer-128', 'num_lstm_layers': 1, 'lstm_units': 128},
    {'name': 'lstm-2layer-128', 'num_lstm_layers': 2, 'lstm_units': 128},
    {'name': 'lstm-3layer-128', 'num_lstm_layers': 3, 'lstm_units': 128},
    {'name': 'lstm-1layer-512', 'num_lstm_layers': 1, 'lstm_units': 512},
    {'name': 'lstm-2layer-512', 'num_lstm_layers': 2, 'lstm_units': 512},
    {'name': 'lstm-3layer-512', 'num_lstm_layers': 3, 'lstm_units': 512},
]

def decode_keras(keras_model, feat_vec, max_len=MAX_LEN):
    start_idx = vocab.get('<start>', 1)
    end_idx   = vocab.get('<end>',   2)
    pad_idx   = vocab.get('<pad>',   0)

    token_ids = [start_idx]
    feat_in   = feat_vec[np.newaxis, :]

    for _ in range(max_len):
        cap_in  = np.array(token_ids + [pad_idx] * (max_len - 1 - len(token_ids)),
                           dtype=np.int32)[np.newaxis, :]
        logits  = keras_model.predict([feat_in, cap_in], verbose=0)
        next_id = int(np.argmax(logits[0, len(token_ids) - 1, :]))
        if next_id == end_idx or next_id == pad_idx:
            break
        token_ids.append(next_id)

    words = [idx2word.get(i, '<unk>') for i in token_ids[1:]]
    return [w for w in words if w not in ('<start>', '<end>', '<pad>', '<unk>')]

def evaluate_model(decoder, test_ids, img_idx, features,
                    test_captions, max_len=MAX_LEN, use_keras=False, keras_model=None):
    references, hypotheses = [], []

    for img_id in test_ids:
        if img_id not in img_idx:
            continue
        refs = [r.split() for r in test_captions.get(img_id, [])]
        if not refs:
            continue
        feat_vec = features[img_idx[img_id]]

        if use_keras:
            hyp = decode_keras(keras_model, feat_vec, max_len)
        else:
            hyp = decoder.generate_caption(feat_vec, max_len).split()

        references.append(refs)
        hypotheses.append(hyp)

    b4  = bleu4(references, hypotheses)
    met = meteor(references, hypotheses)
    return b4, met, references, hypotheses

## 3. Evaluasi Semua Variasi (From Scratch)

In [ ]:
results = {}

for cfg in CONFIGS:
    ckpt = os.path.join(MODELS_DIR, f'{cfg["name"]}.weights.h5')
    if not os.path.exists(ckpt):
        print(f'[SKIP] {cfg["name"]} — checkpoint tidak ditemukan')
        continue

    keras_model = build_lstm_decoder(
        vocab_size=VOCAB_SIZE,
        feature_dim=FEATURE_DIM,
        embed_dim=EMBED_DIM,
        lstm_units=cfg['lstm_units'],
        num_lstm_layers=cfg['num_lstm_layers'],
        max_len=MAX_LEN - 1,
    )
    keras_model.load_weights(ckpt)

    scratch_decoder = LSTMDecoder()
    scratch_decoder.load_weights(keras_model, vocab)

    t0 = time.time()
    b4, met, refs, hyps = evaluate_model(
        scratch_decoder, test_ids, img_idx, features, test_captions, MAX_LEN
    )
    t_scratch = time.time() - t0

    results[cfg['name']] = {
        'bleu4': b4, 'meteor': met,
        'time_scratch': t_scratch,
        'refs': refs, 'hyps': hyps,
    }
    print(f'  BLEU-4: {b4:.4f} | METEOR: {met:.4f} | Time: {t_scratch:.1f}s')

## 4. Perbandingan Keras vs From Scratch (Best Model)

In [ ]:
best_name = max(results, key=lambda k: results[k]['bleu4'])
best_cfg  = next(c for c in CONFIGS if c['name'] == best_name)
print(f'Model terbaik: {best_name} (BLEU-4: {results[best_name]["bleu4"]:.4f})')

keras_model = build_lstm_decoder(
    vocab_size=VOCAB_SIZE, feature_dim=FEATURE_DIM, embed_dim=EMBED_DIM,
    lstm_units=best_cfg['lstm_units'], num_lstm_layers=best_cfg['num_lstm_layers'],
    max_len=MAX_LEN - 1,
)
keras_model.load_weights(os.path.join(MODELS_DIR, f'{best_name}.weights.h5'))

t0 = time.time()
b4_keras, met_keras, _, _ = evaluate_model(
    None, test_ids[:100], img_idx, features, test_captions, MAX_LEN,
    use_keras=True, keras_model=keras_model,
)
t_keras = time.time() - t0

scratch_best = LSTMDecoder()
scratch_best.load_weights(keras_model, vocab)
t0 = time.time()
b4_scratch, met_scratch, _, _ = evaluate_model(
    scratch_best, test_ids[:100], img_idx, features, test_captions, MAX_LEN
)
t_scratch = time.time() - t0

print(f'\nKeras  — BLEU-4: {b4_keras:.4f}, METEOR: {met_keras:.4f}, Time: {t_keras:.1f}s')
print(f'Scratch — BLEU-4: {b4_scratch:.4f}, METEOR: {met_scratch:.4f}, Time: {t_scratch:.1f}s')

## 5. Tabel Perbandingan Semua Variasi

In [ ]:
print(f'{"Variasi":<22} {"Layers":<8} {"Units":<8} {"BLEU-4":<10} {"METEOR":<10} {"Time(s)":<10}')
print('-' * 70)
for cfg in CONFIGS:
    name = cfg['name']
    if name not in results:
        continue
    r = results[name]
    print(f'{name:<22} {cfg["num_lstm_layers"]:<8} {cfg["lstm_units"]:<8}'
          f' {r["bleu4"]:<10.4f} {r["meteor"]:<10.4f} {r["time_scratch"]:<10.1f}')

## 6. Plot BLEU-4 per Variasi

In [ ]:
names  = [cfg['name'] for cfg in CONFIGS if cfg['name'] in results]
bleus  = [results[n]['bleu4']  for n in names]
meteors= [results[n]['meteor'] for n in names]

x = np.arange(len(names))
width = 0.35
fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar(x - width/2, bleus,   width, label='BLEU-4',  color='steelblue')
bars2 = ax.bar(x + width/2, meteors, width, label='METEOR',  color='coral')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=30, ha='right')
ax.set_ylabel('Score')
ax.set_title('BLEU-4 dan METEOR — Semua Variasi LSTM')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../../models/lstm/eval_scores.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Qualitative Analysis — 10 Contoh Gambar

Pilih gambar dengan skor tinggi, sedang, dan rendah.

In [ ]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

smoothie = SmoothingFunction().method4

best_refs = results[best_name]['refs']
best_hyps = results[best_name]['hyps']
valid_test_ids = [img_id for img_id in test_ids if img_id in img_idx
                  and test_captions.get(img_id)]

per_img_bleu = []
for i, (refs, hyp) in enumerate(zip(best_refs, best_hyps)):
    score = sentence_bleu(refs, hyp, weights=(0.25,)*4, smoothing_function=smoothie)
    per_img_bleu.append((score, i))

per_img_bleu.sort(key=lambda x: x[0], reverse=True)

n = len(per_img_bleu)
selected_indices = (
    [per_img_bleu[i][1] for i in range(min(3, n))] +
    [per_img_bleu[i][1] for i in range(n//2 - 2, n//2 + 2)] +
    [per_img_bleu[i][1] for i in range(max(n-3, 0), n)]
)[:10]

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for ax, idx in zip(axes, selected_indices):
    img_id  = valid_test_ids[idx]
    score   = per_img_bleu[0][0] if idx == per_img_bleu[0][1] else next(
        s for s, i in per_img_bleu if i == idx
    )
    hyp_str = ' '.join(best_hyps[idx])
    ref_str = ' '.join(best_refs[idx][0])

    img_path = os.path.join(IMAGES_DIR, img_id)
    if os.path.exists(img_path):
        img = Image.open(img_path).resize((224, 224))
        ax.imshow(img)
    ax.axis('off')
    ax.set_title(
        f'BLEU-4: {score:.3f}\nHyp: {hyp_str[:50]}...\nRef: {ref_str[:50]}...',
        fontsize=7, wrap=True
    )

plt.suptitle(f'Qualitative Analysis — {best_name}', fontsize=13)
plt.tight_layout()
plt.savefig('../../models/lstm/qualitative.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Pengaruh Max Caption Length

Minimal 3 variasi panjang caption.

In [ ]:
MAX_LEN_VARIANTS = [10, 20, 30, 40]
len_results = {}

for max_l in MAX_LEN_VARIANTS:
    b4_l, _, _, _ = evaluate_model(
        scratch_best, test_ids[:200], img_idx, features, test_captions, max_l
    )
    len_results[max_l] = b4_l
    print(f'max_len={max_l:3d} → BLEU-4: {b4_l:.4f}')

plt.figure(figsize=(7, 4))
plt.plot(list(len_results.keys()), list(len_results.values()),
         marker='o', color='steelblue', linewidth=2)
plt.xlabel('Max Caption Length')
plt.ylabel('BLEU-4')
plt.title('Pengaruh Max Caption Length terhadap BLEU-4')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../../models/lstm/max_len_effect.png', dpi=150)
plt.show()

## 9. Batch Inference Demo

In [ ]:
test_sample_ids = test_ids[:5]
test_feats = np.stack([features[img_idx[i]] for i in test_sample_ids
                        if i in img_idx])

single_caps = [scratch_best.generate_caption(test_feats[i]) for i in range(len(test_feats))]

batch_caps  = scratch_best.generate_captions_batch(test_feats, batch_size=16)

print('Single vs Batch inference:')
for i, (s, b) in enumerate(zip(single_caps, batch_caps)):
    match = 'OK' if s == b else 'MISMATCH'
    print(f'  [{match}] {s[:60]}')

N_BENCH = 100
bench_feats = np.stack([features[img_idx[i]] for i in test_ids[:N_BENCH]
                          if i in img_idx])

t0 = time.time()
for fv in bench_feats:
    scratch_best.generate_caption(fv)
t_single = time.time() - t0

t0 = time.time()
scratch_best.generate_captions_batch(bench_feats, batch_size=16)
t_batch = time.time() - t0

print(f'\nSingle loop ({N_BENCH} gambar): {t_single:.2f}s')
print(f'Batch inference ({N_BENCH} gambar, batch_size=16): {t_batch:.2f}s')
print(f'Speedup: {t_single/t_batch:.2f}x')